# Democracy and Epistocracy, Deliberation and Aggregation
This notebook contains code that supports the analysis of different political institutions, ranging from democratic to epistocratic and from deliberation to aggregation. 

## Setup

In [2]:
import os
from math import comb
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats

# from data_analysis.statistics import produce_df_1samp, produce_df_paired
from figures.generate_heatmap import heatmap
from models.sources import Sources
from models.agent import Agent
from models.team import Team
from models.crowd import Crowd
from models.generate_teams import (
    generate_diverse_team,
    generate_expert_team,
    generate_restricted_team,
    generate_random_crowd,
    generate_qualified_heuristics,
)

from utils.basic_functions import (
    powerset,
    calculate_competence_with_duplicates
)

from figures.generate_df import (
    produce_df_results, 
    aggregate_df_results
)

from tqdm.auto import tqdm

In [6]:
comb(17, 5)

6188

In [7]:
comb(13, 5)

1287

# Diversity

In [11]:
def greedy_procedure_maximal_balance(
        n_simulations: int, 
        n_sources: int, 
        heuristic_size: int, 
        team_size: int, 
        progress: bool = True,
    ) -> int:
    n_failures = 0
    iterator = range(n_simulations)
    if progress:
        iterator = tqdm(iterator)
    for _ in iterator:
        team = generate_diverse_team(
            sources=Sources(n_sources),
            heuristic_size=heuristic_size,
            size=team_size,
        )
        source_frequencies = Counter(source for agent in team.members for source in agent.heuristic)
        max_fequency = max(source_frequencies.values())
        min_frequency = min(source_frequencies.values())
        if max_fequency - min_frequency > 1:
            n_failures += 1
            if progress:
                print("Failed due to imbalance")
    return n_failures

In [12]:
greedy_procedure_maximal_balance(
    10_000, 13, 5, 9
)

100%|██████████| 10000/10000 [00:48<00:00, 204.56it/s]


0

In [13]:
greedy_procedure_maximal_balance(
    10_000, 17, 5, 9
)

100%|██████████| 10000/10000 [04:20<00:00, 38.45it/s]


0

# Permissive deliberation

In this section, we perform simulations of the greedy search procedure for permissive teams to determine whether they cover all sources when the epistemic threshold is at the 75th percentile. The result is that permissive teams always cover all sources and, consequently, that permissive deliberation is epistemically equivalent to democratic deliberation. 

In [20]:
def get_best_agent_with_worst_source(n_sources: int, heuristics_size: int):
    sources = Sources(n_sources)
    n_possible_heuristics = comb(n_sources, heuristics_size)
    team_all = generate_expert_team(sources, heuristics_size, n_possible_heuristics)
    k_best_agent = 0
    for k, agent in enumerate(team_all.members):
        if 0 in agent.heuristic:
            k_best_agent = k
            break
    return k_best_agent / team_all.size , team_all.members[k_best_agent]


In [21]:
get_best_agent_with_worst_source(13, 5)

(0.03263403263403263, <models.agent.Agent at 0x1c75b4ded60>)

In [22]:
get_best_agent_with_worst_source(17, 5)

(0.024240465416936006, <models.agent.Agent at 0x1c75b5337e0>)

In [33]:
def simulate_restricted_team_generation(
        n_simulations: int, 
        n_sources: int, 
        heuristic_size: int, 
        qualifying_percentile: float, 
        team_size: int, 
        progress: bool = True,
    ) -> int:
    n_failures = 0
    iterator = range(n_simulations)
    if progress:
        iterator = tqdm(iterator)
    for _ in iterator:
        team = generate_restricted_team(
            sources=Sources(n_sources),
            heuristic_size=heuristic_size,
            qualifying_percentile=qualifying_percentile,
            size=team_size,
        )
        sources_covered = set(source for agent in team.members for source in agent.heuristic)
        if len(sources_covered) < n_sources:
            n_failures += 1
            if progress:
                print("Failed")
    return n_failures

In [34]:
simulate_restricted_team_generation(1_000, 13, 5, 75, 9, progress=True)

100%|██████████| 1000/1000 [02:29<00:00,  6.69it/s]


0

In [35]:
simulate_restricted_team_generation(1_000, 17, 5, 75, 9, progress=True)

100%|██████████| 1000/1000 [11:58<00:00,  1.39it/s]


0

# Weighted procedures

In [14]:
from figures.comparison_weighted_procedure import table_comparison_weighted_procedure

In [16]:
df_comparison = table_comparison_weighted_procedure()
df_comparison

Calculating weighted results: 100%|██████████| 72/72 [00:33<00:00,  2.15it/s]


,n_sources,reliability,group_type,simulated_accuracy,weighted_accuracy,difference
0,13,0.55,restricted_10,0.644837,0.644837,0.000000
1,13,0.55,restricted_20,0.651401,0.652527,-0.001126
2,13,0.55,restricted_30,0.666578,0.666722,-0.000143
3,13,0.55,restricted_40,0.678100,0.679680,-0.001581
4,13,0.55,restricted_50,0.685723,0.687317,-0.001594
5,13,0.55,restricted_60,0.692823,0.693829,-0.001006
6,13,0.55,restricted_70,0.698533,0.700083,-0.001550
7,13,0.55,restricted_80,0.704926,0.706496,-0.001570
8,13,0.55,restricted_90,0.711699,0.712773,-0.001074
9,13,0.60,restricted_10,0.773033,0.773033,0.000000


In [17]:
print(df_comparison["difference"].abs().max())

0.0015941266572009338


In [18]:
print(df_comparison["difference"].abs().median())

0.0007762662232989603
